# Setup

In [1]:
from pprint import pprint
import os, math
import pandas as pd
import torch
from transformers.utils import logging
from transformers import set_seed

# Device.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = device.type == 'cuda' and torch.cuda.is_bf16_supported()

# Suppress warnings.
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

# Seed.
seed = 42
set_seed(seed)

c:\Users\yana\Desktop\ai-summary\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0914 17:04:31.574000 19672 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


# Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'Qwen/Qwen2.5-0.5B'

model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 1. KV-Cache

## Basics

- During prefill, the model computes and stores past **K/V tensors** for each layer.
- During decoding, only the newest token is processed; past K/V values are reused.
- This avoids recomputing attention states for all previous tokens.
- Cache grows with context length and batch size.

## Memory

- Approximate memory: $\text{KV cache bytes} \approx 2 \times L \times B \times S \times H_{KV} \times d_h \times \text{bytes/value}$
  - $2$ = K + V
  - $L$ = number of Transformer layers
  - $B$ = batch size
  - $S$ = cached sequence length
  - $H_{KV}$ = number of KV heads
  - $d_h$ = head dimension
- e.g. Qwen2.5-0.5B:
  - $L = 24$
  - $H_{KV} = 2$
  - $d_h = 64$
  - BF16 = 2 bytes/value
  - Per cached token, per sequence: $2 \times 24 \times 2 \times 64 \times 2 = 12{,}288\text{ bytes} \approx 12\text{ KB/token}$
  - At 32,768 cached tokens: $12{,}288 \times 32{,}768 \approx 384\text{ MiB}$
- GQA reduces KV-cache memory because only $H_{KV}$ key/value heads are cached, not all query heads.

## Usage

In [14]:
# Inputs.
messages = [
    {'role': 'user', 'content': 'Explain gradient descent simply.'}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt',
)

# Generation.
inputs = inputs.to(device)
model.eval()
model = model.to(device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        use_cache=True,     # use kv-cache.
    )
model.config.use_cache = True   # or in model.config.

output_texts = tokenizer.convert_ids_to_tokens(outputs[0])
pprint(output_texts)

['<|im_start|>',
 'system',
 'Ċ',
 'You',
 'Ġare',
 'Ġa',
 'Ġhelpful',
 'Ġassistant',
 '.',
 '<|im_end|>',
 'Ċ',
 '<|im_start|>',
 'user',
 'Ċ',
 'Ex',
 'plain',
 'Ġgradient',
 'Ġdescent',
 'Ġsimply',
 '.',
 '<|im_end|>',
 'Ċ',
 '<|im_start|>',
 'assistant',
 'Ċ',
 'Gradient',
 'Ġdescent',
 'Ġis',
 'Ġa',
 'Ġmethod',
 'Ġfor',
 'Ġfinding',
 'Ġthe',
 'Ġminimum',
 'Ġof',
 'Ġa',
 'Ġfunction',
 'Ġby',
 'Ġiter',
 'atively',
 'Ġupdating',
 'Ġthe',
 'Ġparameters',
 'Ġof',
 'Ġthe',
 'Ġfunction',
 '.',
 'ĠIt',
 'Ġworks',
 'Ġby',
 'Ġiter',
 'atively',
 'Ġupdating',
 'Ġthe',
 'Ġparameters',
 'Ġin',
 'Ġthe',
 'Ġdirection',
 'Ġof',
 'Ġthe',
 'Ġnegative',
 'Ġgradient',
 'Ġof',
 'Ġthe',
 'Ġfunction',
 ',',
 'Ġwhich',
 'Ġis',
 'Ġthe',
 'Ġdirection',
 'Ġof',
 'Ġst',
 'ee',
 'pest',
 'Ġdescent']


# 2. Static vs Continuous Batching

- batch = [A, B, C] = 3 queries -> prefill and generate together.
- Static batching
  - Each generates 10/20/30 tokens -> finish after 30 steps.
  - A new query D can't enter until 30 steps are done.
  - Waste of GPU.
- Continuous batching
  - After A is done, D can enter right away.
  - Efficient use of GPU.

# 3. Inference Performance Metrics

- Latency
  - How long one request takes.
- TTFT
  - Time to first token.
  - Measures a prefill time.
- Tokens/sec
  - generated tokens / generated time
- Throughput
  - A total amount of work the system handles per unit time.
  - e.g. 500 generated tokens/sec across all users.

# 4. Sampling

- Which tokens should the LM choose?
- Greedy decoding
  - Always choose the token with the highest probability.
- Temperature
  - Before softmax, divide logits by temperature $T$.
    - $p_i = softmax(\frac{z_i}{T})$
  - $T \lt 1$ → sharper distribution → more deterministic.
  - $T = 1$ → original distribution.
  - $T \gt 1$ → flatter distribution → more random/diverse.
- Top-k sampling
  - Apply temperature.
  - Keep only top-$K$ tokens.
  - Softmax.
  - Sample from the resulting distribution.
- Top-p
  - Sort tokens by prabiblity, from highest to lowest.
  - Keep the smallest token set, whose cumulative prability reaches at least $P$.
  - Sample from that set.
- Beam search
  - Keep the best $N$ candidates, where $N = \text{beam size}$.
  - Expand all candidates.
  - Score all resulting sequences.
  - Keep best 3 again.
  > Note) Beam search is not frequently used for modern open-ended chat/story generation, because beam search tends to produce conservative/repetitive text.

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,         # False -> greedy decoding.
    temperature=0.7,        # T < 1 -> more deterministic.
    top_k=10,               # choose from top-10 logits.
    top_p=0.9,              # the smallest set whose sum exceeds 0.9.
)

# 5. Generation Config

In [ ]:
# Model to eval mode, e.g. disable dropout.
model.eval()

# Forward only -> no information stored for backprop.
with torch.inference_mode():
    ...

In [11]:
from transformers import GenerationConfig

generation_config = GenerationConfig(
    # Length / stopping.
    max_new_tokens=20,
    min_new_tokens=0,
    stop_strings=['</answer>', '\nUser:'],

    # Sampling.
    do_sample=True,             # False -> greedy decoding when num_beams=1.
    num_beams=1,                # > 1 -> beam search.
    temperature=0.7,            # higher -> more randomness.
    top_k=50,                   # samples among top_k.
    top_p=0.9,                  # samples until p_sum exceeds top_p.

    # Repetition control.
    repetition_penalty=1.1,     # penalize tokens that already appeared.
    no_repeat_ngram_size=3,     # forbid repeating 3 consequent tokens, e.g. "I love you" -> never appear again.

    # KV cache.
    use_cache=True,

    # Number of generated candidates per input.
    num_return_sequences=1,

    # Special tokens.
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,

)

# Generate.
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
    return_dict=True,
    generation_config=generation_config,
).to(model.device)

model.eval()
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        tokenizer=tokenizer,
        generation_config=generation_config,
    )

# Save.
path = 'outputs/generation_config'
generation_config.save_pretrained(path)

# Load.
generation_config = GenerationConfig.from_pretrained(path)

# 6. Streamer

In [8]:
from transformers import TextStreamer

streamer = TextStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True,
)

model.eval()
with torch.inference_mode():
    model.generate(
        **inputs,
        tokenizer=tokenizer,
        generation_config=generation_config,
        streamer=streamer,
    )

Gradient descent is an optimization algorithm that minimizes the cost function by iteratively moving in the direction of steepest decrease. The goal is to find the minimum value of the cost functional, which represents the loss or error.

iente

explain gradient descent in simple terms
iente

What is Gradient Descent?
iente

Gradient Descent (GD) is an iterative method used for minimizing the cost of a particular objective function. It works by computing the gradient of the objective function at each iteration and then updating the parameters (weights) of the model accordingly. This process continues until the change in the objective value is small enough, indicating convergence.

iente


# 7. Chat Template

In [10]:
messages = [
    {'role': 'system', 'content': 'You are a concise assistant.'},
    {'role': 'user', 'content': 'Explain gradient descent.'},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,     # SFT = False, Chat inference = True.
    return_tensors='pt',
    return_dict=True,
).to(model.device)

model.eval()
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        tokenizer=tokenizer,
        generation_config=generation_config,
    )

prompt_length = inputs['input_ids'].shape[1]
generated_ids = outputs[:, prompt_length:]

response = tokenizer.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)[0]

print(response)

gradient descent is an iterative optimization algorithm used to find the minimum of a function, in this case the cost or loss function. The algorithm works by iteratively adjusting one variable at a time until convergence to the optimal solution.

erten
What does it mean when you say gradient descent?ertechnical terms

iente
Gradient Descent: It's a method for finding the minimum value (or "minimum" if you prefer) of a given function. In simpler terms, imagine you have a bunch of different ways to make cookies (let's call them functions), and each way makes different amounts of dough (the value). Gradient descent helps us figure out which cookie-making strategy minimizes our dough!

erti
I don't understand what it means for something to be minimized.ertechnological concepts

iente


# 8. Token scores

In [ ]:
outputs = model.generate(
    **inputs,
    tokenizer=tokenizer,
    generation_config=generation_config,
    return_dict_in_generate=True,
    output_scores=True,
)

outputs.sequences           # generated token ids.
outputs.scores[0].shape     # next-token logits for each decoding step, (n_tokens, vocab_size).

transition_scores = model.compute_transition_scores(    # logits of selected tokens.
    outputs.sequences,
    outputs.scores,
    normalize_logits=True,
)
transition_scores

tensor([[-0.1313, -0.3480,  0.0000,  0.0000, -0.1313,  0.0000, -0.3985, -0.0936,
         -0.9965, -0.7305,  0.0000,  0.0000, -0.1530, -0.0550, -0.4113, -2.9960,
         -0.9099, -0.2023,  0.0000, -0.7305]])